# 05 — Evaluation: Reproducible Paper-Quality Runs

노트북은 실행 순서만 정의하고, 핵심 로직은 `pcdp/evaluation.py`, `pcdp/phase.py`, `pcdp/experiment_plots.py`에 둔다.

**평가 항목**
- Table 1: Vanilla / Periodic / Trajectory in-distribution gait quality (`survival`, `total reward`, `reward/step`, `forward velocity`)
- Table 2: Periodic / Trajectory frequency command tracking (`freq_cmd`, `zone`, `model`, `freq_meas`, `|freq_err|`, `freq_ratio`, `PLV`, `reward/step`, `survival`)
- Figure 1: Vanilla / Periodic / Trajectory reward-per-step comparison
- Figure 2: Reward-per-step vs commanded frequency (phase-conditioned models only)
- Figure 3: Commanded vs measured gait frequency
- Figure 4: Zone-aggregated frequency error and PLV
- Raw arrays: `eval_results.npz`


## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Imports + Config


In [ ]:
import gymnasium as gym
import torch

from pcdp.dataset import load_project_data
from pcdp.evaluation import (
    build_eval_results_payload,
    build_frequency_sweep_protocol,
    load_evaluation_state,
    print_frequency_sweep_summary,
    print_table1_summary,
    run_frequency_sweep_evaluation,
    write_frequency_tracking_table_markdown,
    write_table1_summary_markdown,
    run_in_distribution_evaluation,
    save_eval_results_npz,
)
from pcdp.experiment_plots import plot_paper_figures
from pcdp.configs import set_global_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')


## 3. 데이터 및 체크포인트 로드

In [ ]:
data = load_project_data(DATA_DIR)
set_global_seed(data['seed'], deterministic=True)


In [ ]:
state = load_evaluation_state(data, device=device, checkpoints_dir=CHECKPOINTS_DIR)


## 4. 평가 프로토콜 정의

In [ ]:
DT = 0.05
MAX_STEPS = 1000
N_SEEDS_INDIST = 20
N_SEEDS_SWEEP = 10

freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)

print(f"In-dist freqs: {freq_protocol.in_freqs.round(3).tolist()}")
print(f"OOD freqs:     {freq_protocol.ood_freqs.round(3).tolist()}")
print(f"Sweep freqs:   {freq_protocol.sweep_freqs.round(3).tolist()}")
print(f"Zones:         {freq_protocol.zone_labels.tolist()}")


## 5. Table 1 — In-distribution performance

In [ ]:
env = gym.make('Ant-v5')
table1_results = run_in_distribution_evaluation(
    state,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_INDIST,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_table1_summary(state, table1_results, freq_hz=float(data['freq_window_mean']))
write_table1_summary_markdown(
    state,
    table1_results,
    RESULTS_DIR / 'table1_indist_quality.md',
    freq_hz=float(data['freq_window_mean']),
    interval='ci95',
)


## 6. Table 2 — Frequency command tracking

In [ ]:
freq_results = run_frequency_sweep_evaluation(
    state,
    freq_protocol,
    env=env,
    data=data,
    device=device,
    n_seeds=N_SEEDS_SWEEP,
    max_steps=MAX_STEPS,
    dt=DT,
)


In [ ]:
print_frequency_sweep_summary(data, freq_protocol, freq_results)
write_frequency_tracking_table_markdown(
    freq_protocol,
    freq_results,
    RESULTS_DIR / 'table2_frequency_tracking.md',
    interval='ci95',
)


## 7. Figures — Contribution-focused visualizations

In [ ]:
plot_paper_figures(
    table1_results,
    freq_results,
    freq_protocol,
    data,
    FIGURES_DIR,
    n_seeds_sweep=N_SEEDS_SWEEP,
    interval='ci95',
)


## 8. 결과 저장 — `eval_results.npz`

In [ ]:
eval_payload = build_eval_results_payload(
    data,
    table1_results,
    freq_protocol,
    freq_results,
    n_seeds_indist=N_SEEDS_INDIST,
    n_seeds_sweep=N_SEEDS_SWEEP,
)
save_eval_results_npz(eval_payload, RESULTS_DIR / 'eval_results.npz')


## 9. 완료 체크
- 세 모델 동일 protocol로 측정
- Frequency command-tracking metric을 저장
- Notebook은 orchestration만 담당하고 재사용 로직은 `src`에 위치
